<a href="https://colab.research.google.com/github/anacvetkoo/MOS-projekt/blob/main/notebooks/Scorecard_priprava_RV3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Priprava podatkov College Scorecard za RV3

V tem zvezku sem pripravila zgodovinske podatke o povprečnih neto cenah in skupnih stroških študija na univerzah Ivy League.

Uporabila sem 20 institucionalnih datotek College Scorecard za obdobje 2004–2023.

Iz posameznih datotek sem izbrala osem univerz Ivy League in preverila, za katera leta so podatki dejansko na voljo.

Pripravljene podatke bom uporabila pri nadaljnji analizi RV3 v glavnem zvezku `Ziva_teden2.ipynb`.

Ugotovila sem, da so podatki o neto cenah in skupnih stroških za vseh osem univerz na voljo od leta 2009. Zato sem pripravila končno zbirko za obdobje 2009–2023.

In [1]:
import pandas as pd
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

mapa = Path("/content/drive/MyDrive/FERI/3. letnik/1. semester/MOS/Projekt_teden3")
mapa_scorecard = mapa / "College_Scorecard_zgodovinski"

ivy_unitid = [217156, 190150, 190415, 182670, 166027, 186131, 215062, 130794]

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
# Branje podatkov za obdobje 2004–2023

podatki = []

for leto in range(2004, 2024):
    datoteka = mapa_scorecard / f"MERGED{leto}_{str(leto + 1)[-2:]}_PP.csv"

    df = pd.read_csv(
        datoteka,
        usecols=lambda x: x in ["UNITID", "NPT4_PRIV", "COSTT4_A"],
        low_memory=False
    )

    df = df[df["UNITID"].isin(ivy_unitid)].copy()

    # Manjkajoči stolpci v starejših datotekah
    for stolpec in ["NPT4_PRIV", "COSTT4_A"]:
        if stolpec not in df.columns:
            df[stolpec] = pd.NA
        df[stolpec] = pd.to_numeric(df[stolpec], errors="coerce")

    df["leto"] = leto
    podatki.append(df)

# Združevanje vseh let
df_scorecard_ivy = pd.concat(podatki, ignore_index=True)

display(df_scorecard_ivy.head())

,UNITID,NPT4_PRIV,COSTT4_A,leto
0,130794,NaN,NaN,2004
1,166027,NaN,NaN,2004
2,182670,NaN,NaN,2004
3,186131,NaN,NaN,2004
4,190150,NaN,NaN,2004


In [3]:
# Pregled razpoložljivosti podatkov po letih
pregled = df_scorecard_ivy.groupby("leto").agg(
    univerze=("UNITID", "nunique"),
    neto_cene=("NPT4_PRIV", "count"),
    skupni_stroski=("COSTT4_A", "count")
)

display(pregled)

# Izbor obdobja z razpoložljivimi podatki
df_koncni = df_scorecard_ivy[
    df_scorecard_ivy["leto"] >= 2009
].copy()

# Preverjanje popolnosti podatkov
assert len(df_koncni) == 120
assert not df_koncni.duplicated(["UNITID", "leto"]).any()
assert df_koncni[["NPT4_PRIV", "COSTT4_A"]].notna().all().all()

# Shranjevanje končne zbirke
df_koncni.to_csv(
    mapa / "Scorecard_Ivy_2009_2023.csv",
    index=False
)

print("Podatki uspešno shranjeni:", len(df_koncni), "zapisov.")

,univerze,neto_cene,skupni_stroski
leto,,,
2004,8,0,0
2005,8,0,0
2006,8,0,0
2007,8,0,0
2008,8,0,0
2009,8,8,8
2010,8,8,8
2011,8,8,8
2012,8,8,8


Podatki uspešno shranjeni: 120 zapisov.
